In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Instalar pendências

In [ ]:
!pip install rouge-score openai sentence-transformers scikit-learn pycocoevalcap -q

In [ ]:
!pip install bert-score -q
!pip install sacrebleu -q
!pip install rouge_score -q
!pip install meteor_score -q

In [ ]:
!pip install openai -q

Imports

In [ ]:
import os
import re
import json
import math
import random
import time
from pathlib import Path
from typing import List, Tuple, Dict, Any
from openai import OpenAI

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# NLP libs
import spacy
from spacy.lang.pt.stop_words import STOP_WORDS as SPACY_STOPWORDS

import torch
from transformers import AutoModel, AutoTokenizer
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from rouge_score import rouge_scorer
from bert_score import score as bert_score
from nltk.translate.meteor_score import meteor_score
import sacrebleu

import nltk
nltk.download('wordnet')

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

#keys
HUGGINGFACE_TOKEN = os.environ.get('HF_TOKEN', 'insira_sua_key')
OPENROUTER_KEY = os.environ.get('insira_key', None)

# model
BUMBABERT_MODEL_ID = "LincProgUEMA/BUMBABERT_SC_SMALL"
MODEL_ASSUME_LOWERCASE = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16

In [ ]:
!python -m spacy download pt_core_news_sm

In [ ]:
try:
  nlp = spacy.load("pt_core_news_sm")
except Exception:
  nlp = spacy.blank("pt")

In [ ]:
!huggingface-cli login

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: write).
The token `summ` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as 

Dataset

In [ ]:
!wget -q https://github.com/diego-feijo/rulingbr/raw/master/rulingbr-v1.2.tar.xz
!tar -xf rulingbr-v1.2.tar.xz

In [ ]:
docs = []
with open('rulingbr-v1.2.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            docs.append(json.loads(line))
print(f"Total de documentos: {len(docs)}")

In [ ]:
def count_words(text):
    return len(text.split())

full_text = ' '.join([docs.get('acordao',''), docs.get('voto',''), docs.get('relatorio','')])
n_words = count_words(full_text)
print(n_words)

In [ ]:
from openai import OpenAI

OPENROUTER_KEY = "insira_sua_key"
client = OpenAI(api_key=OPENROUTER_KEY, base_url="https://openrouter.ai/api/v1")

try:
    response = client.chat.completions.create(
        model="google/gemini-2.5-flash-lite",
        messages=[{"role": "user", "content": "Escreva uma frase curta de teste."}],
        max_tokens=256
    )
    print("API funcionando! Resposta:\n", response.choices[0].message.content)
except Exception as e:
    print("Erro na API:", e)

API funcionando! Resposta:
 Claro, aqui está uma frase curta de teste:

"O gato preto saltou."


In [ ]:
def preprocess_text(text: str, lowercase: bool = MODEL_ASSUME_LOWERCASE) -> str:

    if not text:
        return ""
    # whitespace
    text = re.sub(r"\s+", " ", text).strip()
    # remove control characters
    text = re.sub(r"[\r\t\x0b\x0c]", " ", text)
    if lowercase:
        text = text.lower()
    return text

In [ ]:
def sentence_segment(text: str) -> List[str]:
    if not text:
        return []
    doc = nlp(text)
    sents = [sent.text.strip() for sent in doc.sents]
    if len(sents) == 0:
        sents = [s.strip() for s in re.split(r'(?<=[\.\?!])\s+', text) if len(s.strip()) > 10]
    return [s for s in sents if len(s) > 20]


STOPWORDS_PT = set(list(SPACY_STOPWORDS))

In [ ]:
bumbabert_tokenizer = AutoTokenizer.from_pretrained(BUMBABERT_MODEL_ID, use_auth_token=HUGGINGFACE_TOKEN)
bumbabert_model = AutoModel.from_pretrained(BUMBABERT_MODEL_ID, use_auth_token=HUGGINGFACE_TOKEN).to(DEVICE)

In [ ]:
def count_tokens_with_tokenizer(texts: List[str], tokenizer: AutoTokenizer, max_length: int = None) -> List[int]:
    counts = []
    for t in texts:
        enc = tokenizer(
            t,
            add_special_tokens=True,
            truncation=False  # nunca truncar
        )
        n_tokens = len(enc['input_ids'])
        # aplica max_length se fornecido
        if max_length is not None:
            n_tokens = min(n_tokens, max_length)
        counts.append(n_tokens)
    return counts

In [ ]:
def extractive_tfidf_summary(text: str, num_sentences: int = 3) -> str:
    sentences = sentence_segment(text)
    if len(sentences) <= num_sentences:
        return ' '.join(sentences)
    # Vectorize using Portuguese stopwords from spaCy
    vectorizer = TfidfVectorizer(stop_words=list(STOPWORDS_PT), lowercase=MODEL_ASSUME_LOWERCASE)
    tfidf = vectorizer.fit_transform(sentences)
    scores = np.array(tfidf.sum(axis=1)).flatten()
    top_idx = scores.argsort()[-num_sentences:][::-1]
    top_idx.sort()
    return ' '.join([sentences[i] for i in top_idx])

In [ ]:
def batched_embeddings(text: str, tokenizer: AutoTokenizer, model: AutoModel, device: torch.device = DEVICE, max_length: int = 512, batch_size: int = 8) -> np.ndarray:
    model.eval()
    enc_all = tokenizer(text, return_tensors="pt", add_special_tokens=True)
    input_ids = enc_all["input_ids"].squeeze(0)  # (seq_len,)

    blocks = []
    for i in range(0, input_ids.size(0), max_length):
        blocks.append(input_ids[i:i+max_length])

    all_embeddings = []
    for i in range(0, len(blocks), batch_size):
        batch_blocks = blocks[i:i+batch_size]
        # batch tensor
        batch_tensor = torch.nn.utils.rnn.pad_sequence(batch_blocks, batch_first=True, padding_value=tokenizer.pad_token_id)
        # clamp IDs fora do vocabulário
        batch_tensor = torch.clamp(batch_tensor, 0, model.config.vocab_size - 1)
        batch_tensor = batch_tensor.to(device)
        # atenção mask
        attention_mask = (batch_tensor != tokenizer.pad_token_id).long().to(device)
        # forward
        with torch.no_grad():
            outputs = model(input_ids=batch_tensor, attention_mask=attention_mask)
            # CLS token
            cls_emb = outputs.last_hidden_state[:, 0, :]  # (B, hidden)
            all_embeddings.append(cls_emb.cpu())

    # agregar todos os blocos
    all_embeddings = torch.cat(all_embeddings, dim=0)
    doc_embedding = all_embeddings.mean(dim=0).numpy()  # média sobre todos os blocos
    return doc_embedding

In [ ]:
def extractive_bert_summary(text: str, num_sentences: int = 5, tokenizer=bumbabert_tokenizer, model=bumbabert_model, device: torch.device = DEVICE) -> str:
    sentences = sentence_segment(text)
    if len(sentences) <= num_sentences:
        return ' '.join(sentences)

    # embeddings das sentenças
    sent_embs = []
    for sent in sentences:
        sent_emb = batched_embeddings(sent, tokenizer, model, device=device)
        sent_embs.append(sent_emb)
    sent_embs = np.vstack(sent_embs)

    # embedding dos docs
    doc_emb = batched_embeddings(text, tokenizer, model, device=device)
    if doc_emb.ndim == 1:
        doc_emb = doc_emb.reshape(1, -1)

    # similaridade coseno
    sims = cosine_similarity(sent_embs, doc_emb).flatten()

    # seleciona top sentenças
    top_idx = np.argsort(sims)[-num_sentences:][::-1]
    top_idx.sort()
    return ' '.join([sentences[i] for i in top_idx])

In [ ]:
def build_legal_prompt(text: str, ementas_exemplo: List[str], word_limit=(50,100)) -> str:
    """
    Constrói um prompt detalhado para geração de ementas jurídicas.
    - text: conteúdo do processo/acórdão.
    - ementas_exemplo: lista de ementas de referência para estilo e formatação.
    - word_limit: limite de palavras para o corpo da ementa (dispositivo + conclusão).
    """
    prompt = "Você é um especialista em redação de ementas jurídicas concisas e claras em português.\n"
    prompt += "Sua tarefa é transformar o texto jurídico fornecido em uma **ementa jurídica**, seguindo a estrutura padronizada (Cabeçalho, Tese/Dispositivo e Conclusão/Resultado).\n\n"

    prompt += "Siga os exemplos de ementas abaixo para capturar o estilo, a concisão e a formatação desejados:\n\n"

    for i, ementa in enumerate(ementas_exemplo, 1):
        prompt += f"Exemplo {i}: {ementa}\n\n"

    prompt += f"""**Instruções de Estrutura e Formatação**:
1. **CABEÇALHO/VERBETAÇÃO**: Comece com palavras-chave em caixa alta, separadas por pontos finais e sem pontuação final (ex: DIREITO CONSTITUCIONAL. RECURSO EXTRAORDINÁRIO. ICMS).
2. **TESE CENTRAL/DISPOSITIVO**: Redija enunciados completos, concisos e numerados que contenham o entendimento do Tribunal, os fundamentos jurídicos relevantes e as teses firmadas.
3. **CONCLUSÃO/RESULTADO (Opcional)**: Termine com o resultado do julgamento em uma frase final (ex: Recurso desprovido.).

**Entrada para Ementa**:
{text}

**Parâmetros Adicionais**:
- Foque nos pontos centrais do texto, destacando a decisão principal e os fundamentos jurídicos.
- Mantenha coerência, clareza e linguagem formal/técnica.
- Comprimento desejado para o corpo do texto (Dispositivo e Conclusão): {word_limit[0]}-{word_limit[1]} palavras.
- Evite bullet.
- Não inclua opiniões pessoais ou interpretações.

**Resumo**:"""

    return prompt

In [ ]:
def abstractive_api_summary(text: str, examples: List[str], client=None, model_name: str = "google/gemini-2.5-flash-lite", retries=3, delay=5) -> str:
    prompt = build_legal_prompt(text, examples)
    if client is None:
        return text[:512]

    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model_name,
                messages=[{"role":"user","content":prompt}],
                max_tokens=256,
                temperature=0.2
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            print(f"API error (attempt {attempt+1}/{retries}): {e}")
            if attempt < retries - 1:
                time.sleep(delay * (attempt+1))  # backoff linear
            else:
                print("Falha após múltiplas tentativas, retornando fallback.")
                return text[:512]  # fallback simples

In [ ]:
rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
def evaluate_summary(reference: str, generated: str, bert_lang='pt', bert_max_length=512) -> Dict[str, Any]:
    # ROUGE
    r = rouge.score(reference, generated)
    rouge_scores = {k: v.fmeasure for k,v in r.items()}

    # BERTScore
    ref_trunc = reference[:bert_max_length*4]
    gen_trunc = generated[:bert_max_length*4]
    try:
        P, R, F1 = bert_score([gen_trunc], [ref_trunc], lang=bert_lang, rescale_with_baseline=True)
        bert_f1 = float(F1[0].cpu().numpy())
    except Exception as e:
        print("BERTScore error:", e)
        bert_f1 = None

    # METEOR expects (references, hypothesis)
    try:
        reference_tokens = reference.split()
        generated_tokens = generated.split()
        meteor = meteor_score([reference_tokens], generated_tokens)
    except Exception as e:
        print("METEOR error:", e)
        meteor = None

    # BLEU via sacrebleu: sacrebleu.sentence_bleu(hypothesis, [reference])
    try:
        bleu = sacrebleu.sentence_bleu(generated, [reference]).score / 100.0
    except Exception as e:
        print("BLEU error:", e)
        bleu = None

    return {
        **rouge_scores,
        "bertscore": bert_f1,
        "meteor": meteor,
        "bleu": bleu
    }

In [ ]:
def compute_token_stats(texts: List[str], tokenizer: AutoTokenizer, max_length: int = None) -> Dict[str, Any]:
    counts = count_tokens_with_tokenizer(texts, tokenizer, max_length=max_length)
    arr = np.array(counts)
    stats = {
        'n': len(arr),
        'mean': float(arr.mean()),
        'std': float(arr.std()),
        'min': int(arr.min()),
        '25%': int(np.percentile(arr, 25)),
        '50%': int(np.percentile(arr, 50)),
        '75%': int(np.percentile(arr, 75)),
        'max': int(np.max(arr))
    }
    return stats, counts


def load_jsonl(path: str) -> List[dict]:
    out = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            out.append(json.loads(line))
    return out

In [ ]:
def run_experiments_incremental(
    docs: List[dict],
    examples_for_prompt: List[str],
    results_path: str,
    checkpoint_path: str,
    api_client=None,
) -> None:

    start_idx = 0
    if Path(checkpoint_path).exists():
        with open(checkpoint_path, "r") as f:
            start_idx = int(f.read().strip())
        print(f"Retomando do documento {start_idx}...")

    for i, doc in tqdm(enumerate(docs[start_idx:], start=start_idx), total=len(docs) - start_idx):
        try:
            full = preprocess_text(' '.join([doc.get('acordao',''), doc.get('voto',''), doc.get('relatorio','')]))
            ref = preprocess_text(doc.get('ementa',''))
            if len(full.split()) < 50 or len(ref.split()) < 3:
                continue

            tfidf_sum = extractive_tfidf_summary(full, num_sentences=3)
            bert_sum = extractive_bert_summary(full, num_sentences=5)
            gemini_sum = abstractive_api_summary(full, examples_for_prompt, client=api_client)
            hybrid_tfidf_sum = abstractive_api_summary(
                extractive_tfidf_summary(full, num_sentences=8),
                examples_for_prompt,
                client=api_client,
            )
            hybrid_bert_sum = abstractive_api_summary(
                extractive_bert_summary(full, num_sentences=8),
                examples_for_prompt,
                client=api_client,
            )

            tfidf_scores = evaluate_summary(ref, tfidf_sum)
            bert_scores = evaluate_summary(ref, bert_sum)
            gemini_scores = evaluate_summary(ref, gemini_sum)
            htfidf_scores = evaluate_summary(ref, hybrid_tfidf_sum)
            hbert_scores = evaluate_summary(ref, hybrid_bert_sum)

            row = {
                "doc_id": i,
                "reference": ref,
                "tfidf_summary": tfidf_sum,
                "bert_summary": bert_sum,
                "gemini_summary": gemini_sum,
                "hybrid_tfidf_summary": hybrid_tfidf_sum,
                "hybrid_bert_summary": hybrid_bert_sum,
                **{f"tfidf_{k}": v for k,v in tfidf_scores.items()},
                **{f"bert_{k}": v for k,v in bert_scores.items()},
                **{f"gemini_{k}": v for k,v in gemini_scores.items()},
                **{f"hybrid_tfidf_{k}": v for k,v in htfidf_scores.items()},
                **{f"hybrid_bert_{k}": v for k,v in hbert_scores.items()},
            }

            pd.DataFrame([row]).to_csv(
                results_path,
                mode="a",
                header=not Path(results_path).exists(),
                index=False,
            )

            with open(checkpoint_path, "w") as f:
                f.write(str(i+1))

        except Exception as e:
            print(f"Erro no doc {i}: {e}")
            continue

    print("concluído")


In [ ]:
def select_top_examples(df: pd.DataFrame, method: str = 'gemini', metric: str = 'bertscore', top_k: int = 5) -> pd.DataFrame:
    col = f"{method}_{metric}"
    if col not in df.columns:
        raise ValueError(f"Column {col} not in df")
    return df.sort_values(col, ascending=False).head(top_k)

In [ ]:
def plot_metric_boxplots(df: pd.DataFrame, methods: List[str], metrics: List[str]):
    # melt
    cols = [f"{m}_{met}" for m in methods for met in metrics if f"{m}_{met}" in df.columns]
    melted = df.melt(value_vars=cols, var_name='method_metric', value_name='score')
    melted['method'] = melted['method_metric'].apply(lambda x: x.split('_')[0])
    melted['metric'] = melted['method_metric'].apply(lambda x: '_'.join(x.split('_')[1:]))
    plt.figure(figsize=(12,6))
    import seaborn as sns
    sns.boxplot(data=melted, x='metric', y='score', hue='method')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
from openai import OpenAI

if __name__ == '__main__':
    results_path = "/results_summary_pipeline.csv"
    checkpoint_path = "/summarization/checkpoint.txt"
    data_path = "rulingbr-v1.2.jsonl"


    if not Path(data_path).exists():
        raise FileNotFoundError(data_path)
    docs = load_jsonl(data_path)


    docs_valid = [
        d for d in docs
        if len(preprocess_text(' '.join([d.get('acordao',''), d.get('voto',''), d.get('relatorio','')])).split()) > 50
        and len(preprocess_text(d.get('ementa','')).split()) > 5
    ]

    # exemplo pro prompt
    examples = [preprocess_text(d.get('ementa','')) for d in random.sample(docs_valid, 3)]

    #openrouter
    OPENROUTER_KEY = "insira_sua_key"
    client = OpenAI(api_key=OPENROUTER_KEY, base_url="https://openrouter.ai/api/v1")


    run_experiments_incremental(
        docs=docs_valid,
        examples_for_prompt=examples,
        results_path=results_path,
        checkpoint_path=checkpoint_path,
        api_client=client
    )

    df_results = pd.read_csv(results_path)
    print(df_results.head())


    methods = ['tfidf','bert','gemini','hybrid_tfidf','hybrid_bert']
    metrics = ['rouge1','rouge2','rougeL','bertscore','meteor','bleu']
    plot_metric_boxplots(df_results, methods, metrics)

    # top métricas
    try:
        top = select_top_examples(df_results, method='gemini', metric='bertscore', top_k=5)
        top.to_csv(
            "/summarization/top_gemini_examples.csv",
            index=False
        )
    except Exception as e:
        print(e)

    print('fim')

## EDA

In [ ]:
def analyze_data(docs, sample_size=1000):
    sample = docs[:sample_size]
    text_lens, ementa_lens = [], []

    for doc in sample:
        full_text = doc.get('acordao', '') + ' ' + doc.get('voto', '') + ' ' + doc.get('relatorio', '')
        ementa = doc.get('ementa', '')
        text_lens.append(len(full_text.split()))
        ementa_lens.append(len(ementa.split()))

    print(f"Média texto: {np.mean(text_lens):.1f} | Média ementa: {np.mean(ementa_lens):.1f}")
    return sample

sample_docs = analyze_data(docs)